In [12]:
import sys
import urllib.request
from pathlib import Path

URLS = (
    "https://serre-lab.clps.brown.edu/wp-content/uploads/2013/10/hmdb51_org.rar",
    "https://web.archive.org/web/20211105033902id_/https://serre-lab.clps.brown.edu/wp-content/uploads/2013/10/hmdb51_org.rar",
)
DEST = Path("../data") / "hmdb51_org.rar"
RAR_SIGNATURE = b"Rar!\x1a\x07"


def show_progress(downloaded, total_size):
    """Print download progress in the terminal."""
    mb = downloaded / (1024 * 1024)
    if total_size > 0:
        percent = min(100, downloaded * 100 / total_size)
        total_mb = total_size / (1024 * 1024)
        status = f"{percent:5.1f}%  {mb:7.1f} / {total_mb:.1f} MB"
    else:
        status = f"{mb:7.1f} MB downloaded"
    sys.stdout.write(f"\r{status}")
    sys.stdout.flush()


def download_archive(url, destination):
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    partial_path = destination.with_name(f"{destination.name}.part")

    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            content_type = response.headers.get("Content-Type", "").lower()
            if "text/html" in content_type:
                raise ValueError(f"server returned HTML instead of the archive ({response.geturl()})")

            content_length = response.headers.get("Content-Length")
            total_size = int(content_length) if content_length else 0
            downloaded = 0
            with partial_path.open("wb") as output:
                while block := response.read(1024 * 1024):
                    output.write(block)
                    downloaded += len(block)
                    show_progress(downloaded, total_size)

        if total_size and downloaded != total_size:
            raise ValueError(f"incomplete download: received {downloaded} of {total_size} bytes")

        with partial_path.open("rb") as archive:
            if archive.read(len(RAR_SIGNATURE)) != RAR_SIGNATURE:
                raise ValueError("downloaded file is not a RAR archive")

        partial_path.replace(destination)
    finally:
        partial_path.unlink(missing_ok=True)


def main():
    DEST.parent.mkdir(parents=True, exist_ok=True)

    if DEST.exists():
        with DEST.open("rb") as existing_file:
            if existing_file.read(len(RAR_SIGNATURE)) == RAR_SIGNATURE:
                print(f"Archive already exists: {DEST.resolve()}")
                return
        print(f"Existing file is invalid and will be replaced: {DEST.resolve()}")

    errors = []
    for url in URLS:
        print(f"Downloading HMDB51 from {url}...")
        try:
            download_archive(url, DEST)
        except (OSError, ValueError) as error:
            errors.append(f"{url}: {error}")
            print(f"\nDownload failed: {error}")
            continue

        print(f"\nDone! Saved valid archive to: {DEST.resolve()}")
        return

    raise RuntimeError("Unable to download a valid HMDB51 archive:\n" + "\n".join(errors))

main()


Download failed: server returned HTML instead of the archive (https://serre.lab.brown.edu/)
100.0%   2025.6 / 2025.6 MB
Done! Saved valid archive to: /Users/asadbek/Documents/github_projects/sit-jump-recognition/data/hmdb51_org.rar


In [17]:
import csv
import subprocess
from pathlib import Path

import cv2

RAR_PATH = Path("../data") / "hmdb51_org.rar"   # downloaded archive
MAIN_DIR = Path("../data") / "main"             # where the main archive is unpacked
VIDEO_DIR = Path("../data/raw")                   # final videos folder
CLASSES = ["jump", "sit"]
METADATA_PATH = Path("../data") / "video_metadata.csv"

def extract_rar(archive: Path, dest: Path):
    """Extract a .rar file using the `unar` command line tool."""
    dest.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unar", "-f", "-D", "-o", str(dest), str(archive)], check=True)


def main():
    # 1) Unpack the main archive (contains one .rar per class)
    print("Extracting main archive...")
    extract_rar(RAR_PATH, MAIN_DIR)

    # 2) Unpack only the classes we need
    for cls in CLASSES:
        cls_rar = next(MAIN_DIR.rglob(f"{cls}.rar"))
        print(f"Extracting class: {cls}")
        extract_rar(cls_rar, VIDEO_DIR )

    # 3) Read and display metadata for every extracted video
    video_paths = sorted(
        path for path in VIDEO_DIR.rglob("*")
        if path.is_file() and path.suffix.lower() == ".avi"
    )
    METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    fields = [
        "class", "file", "readable", "frames", "fps", "width", "height",
        "duration_seconds", "size_bytes", "error",
    ]

    with METADATA_PATH.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fields)
        writer.writeheader()
        print(f"\nVideo metadata ({len(video_paths)} videos):")

        for path in video_paths:
            capture = cv2.VideoCapture(str(path))
            try:
                row = {
                    "class": path.parent.name,
                    "file": str(path),
                    "readable": capture.isOpened(),
                    "frames": "",
                    "fps": "",
                    "width": "",
                    "height": "",
                    "duration_seconds": "",
                    "size_bytes": path.stat().st_size,
                    "error": "",
                }
                if capture.isOpened():
                    frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
                    fps = capture.get(cv2.CAP_PROP_FPS)
                    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
                    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
                    duration = frames / fps if fps > 0 else None
                    row.update({
                        "frames": frames,
                        "fps": round(fps, 3),
                        "width": width,
                        "height": height,
                        "duration_seconds": round(duration, 3) if duration is not None else "",
                    })
                    duration_text = f"{duration:.2f} s" if duration is not None else "unknown duration"
                    print(
                        f"{path} | {frames} frames | {fps:.2f} FPS | "
                        f"{width}x{height} | {duration_text} | "
                        f"{path.stat().st_size / (1024 * 1024):.2f} MB"
                    )
                else:
                    row["error"] = "OpenCV could not open this video"
                    print(f"{path} | ERROR: {row['error']}")
                writer.writerow(row)
            finally:
                capture.release()

    print(f"\nMetadata for all videos saved to: {METADATA_PATH.resolve()}")


main()

Extracting main archive...
../data/hmdb51_org.rar: RAR
  shoot_gun.rar  (32372358 B)... OK.
  sit.rar  (41022787 B)... OK.
  situp.rar  (27753586 B)... OK.
  smile.rar  (17963800 B)... OK.
  smoke.rar  (43034968 B)... OK.
  somersault.rar  (35618370 B)... OK.
  stand.rar  (38862951 B)... OK.
  swing_baseball.rar  (58308188 B)... OK.
  sword.rar  (35555921 B)... OK.
  sword_exercise.rar  (38602974 B)... OK.
  talk.rar  (52343293 B)... OK.
  throw.rar  (33479582 B)... OK.
  turn.rar  (54143194 B)... OK.
  walk.rar  (177522918 B)... OK.
  wave.rar  (29439358 B)... OK.
  brush_hair.rar  (83671610 B)... OK.
  cartwheel.rar  (30689539 B)... OK.
  catch.rar  (17989454 B)... OK.
  chew.rar  (33328463 B)... OK.
  clap.rar  (33579036 B)... OK.
  climb.rar  (55571348 B)... OK.
  climb_stairs.rar  (34598893 B)... OK.
  dive.rar  (35254666 B)... OK.
  draw_sword.rar  (24393246 B)... OK.
  dribble.rar  (49973413 B)... OK.
  drink.rar  (43571726 B)... OK.
  eat.rar  (28848032 B)... OK.
  fall_floor.r

In [20]:
import pandas as pd

df = pd.read_csv("../data/video_metadata.csv")

In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 293 entries, 0 to 292
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   class             293 non-null    str    
 1   file              293 non-null    str    
 2   readable          293 non-null    bool   
 3   frames            293 non-null    int64  
 4   fps               293 non-null    float64
 5   width             293 non-null    int64  
 6   height            293 non-null    int64  
 7   duration_seconds  293 non-null    float64
 8   size_bytes        293 non-null    int64  
 9   error             0 non-null      float64
dtypes: bool(1), float64(3), int64(4), str(2)
memory usage: 21.0 KB


In [22]:
df.head()

,class,file,readable,frames,fps,width,height,duration_seconds,size_bytes,error
0,jump,../data/raw/jump/50_FIRST_DATES_jump_f_nm_np1_...,True,45,30.0,320,240,1.500,242688,NaN
1,jump,../data/raw/jump/A_Beautiful_Mind_1_jump_f_cm_...,True,68,30.0,352,240,2.267,271360,NaN
2,jump,../data/raw/jump/Arrasando_no_Le_Parkour_jump_...,True,35,30.0,424,240,1.167,131584,NaN
3,jump,../data/raw/jump/Arrasando_no_Le_Parkour_jump_...,True,64,30.0,424,240,2.133,250368,NaN
4,jump,../data/raw/jump/Arrasando_no_Le_Parkour_jump_...,True,48,30.0,424,240,1.600,222208,NaN


In [30]:
import shutil
from pathlib import Path


import cv2
EXTRACTED_FRAMES_DIR= Path("../data") / "extracted_frames"
RAW_VIDEOS_DIR = Path("../data") / "raw"

def reset_directory(path: Path):
    """Remove an old directory and recreate it to keep the pipeline clean."""
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def extract_all_frames(video_path: Path, output_dir: Path):
    """Read every frame from one video and save it as an image."""
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")

    frame_index = 0
    saved_count = 0

    while True:
        success, frame = capture.read()
        if not success:
            break

        output_path = output_dir / f"frame_{frame_index:04d}.jpg"
        cv2.imwrite(str(output_path), frame)
        frame_index += 1
        saved_count += 1

    capture.release()
    return saved_count


def main():
    video_paths = sorted(RAW_VIDEOS_DIR.glob("*/*.*"))
    if not video_paths:
        raise FileNotFoundError(
            f"No videos found in {RAW_VIDEOS_DIR}. Please run download_videos.py first."
        )

    reset_directory(EXTRACTED_FRAMES_DIR)

    for video_path in video_paths:
        class_name = video_path.parent.name
        video_name = video_path.stem
        output_dir = EXTRACTED_FRAMES_DIR / class_name / video_name
        output_dir.mkdir(parents=True, exist_ok=True)

        saved_count = extract_all_frames(video_path, output_dir)
       
        print(f"Extracted {saved_count} frames from {video_path.name}")

    print("\nSaved extraction summary to:", EXTRACTED_FRAMES_DIR / "frame_extraction_summary.json")

main()

Extracted 44 frames from 50_FIRST_DATES_jump_f_nm_np1_fr_bad_3.avi
Extracted 67 frames from A_Beautiful_Mind_1_jump_f_cm_np1_ba_bad_3.avi
Extracted 34 frames from Arrasando_no_Le_Parkour_jump_f_cm_np1_le_bad_0.avi
Extracted 63 frames from Arrasando_no_Le_Parkour_jump_f_cm_np1_ri_bad_1.avi
Extracted 47 frames from Arrasando_no_Le_Parkour_jump_f_nm_np1_le_med_2.avi
Extracted 54 frames from Arrasando_no_Le_Parkour_jump_f_nm_np1_ri_med_3.avi
Extracted 47 frames from BATMAN_BEGINS_jump_f_cm_np1_ri_med_18.avi
Extracted 109 frames from BLACK_HAWK_DOWN_jump_f_cm_np1_ri_bad_8.avi
Extracted 24 frames from BLACK_HAWK_DOWN_jump_f_cm_np1_ri_bad_9.avi
Extracted 49 frames from CharlieAndTheChocolateFactory_jump_f_nm_np1_fr_bad_5.avi
Extracted 168 frames from DONNIE_DARKO_jump_f_nm_np1_fr_med_7.avi
Extracted 74 frames from DONNIE_DARKO_jump_f_nm_np1_ri_bad_6.avi
Extracted 35 frames from Documentario_Le_Parkour_Londrina_jump_f_cm_np1_ba_bad_12.avi
Extracted 55 frames from Documentario_Le_Parkour_Londri

In [29]:
video_paths = sorted(RAW_VIDEOS_DIR.glob("*/*.*"))
video_paths[0].parent.name
video_paths[0].stem

'50_FIRST_DATES_jump_f_nm_np1_fr_bad_3'

In [31]:
import random
import shutil
import numpy as np
import torch



def copy_sequence_frames(source_frame_paths, destination_dir):
    """Copy selected frames into one sequence folder in the correct time order."""
    destination_dir.mkdir(parents=True, exist_ok=True)
    for new_index, frame_path in enumerate(source_frame_paths):
        shutil.copy2(frame_path, destination_dir / f"frame_{new_index:02d}.jpg")


def split_list(items, train_ratio=0.7, val_ratio=0.15):
    total = len(items)
    train_count = max(1, int(total * train_ratio)) if total >= 3 else max(1, total - 2)
    val_count = max(1, int(total * val_ratio)) if total >= 3 else 1

    if train_count + val_count >= total:
        val_count = max(1, total - train_count - 1)

    train_items = items[:train_count]
    val_items = items[train_count:train_count + val_count]
    test_items = items[train_count + val_count:]

    if not test_items and val_items:
        test_items = [val_items.pop()]
    if not val_items and train_items:
        val_items = [train_items.pop()]

    return train_items, val_items, test_items

FINAL_DATA = Path("../data") / "final_data"
SEQUENCES_DIR= FINAL_DATA / "sequences"
TRAIN_DIR = FINAL_DATA / "train"
VAL_DIR = FINAL_DATA / "val"
TEST_DIR = FINAL_DATA / "test"
def sample_frame_indices(total_frames: int, sequence_length: int):
    """
    Pick evenly spaced frame indices.
    This makes every sequence have exactly the same number of frames.
    """
    if total_frames <= 0:
        return [0] * sequence_length

    if total_frames < sequence_length:
        indices = list(range(total_frames))
        while len(indices) < sequence_length:
            indices.append(total_frames - 1)
        return indices

    indices = np.linspace(0, total_frames - 1, sequence_length, dtype=int).tolist()
    return indices

def count_items_per_class(root_dir: Path):
    counts = {}
    if not root_dir.exists():
        return counts
    for class_dir in sorted(root_dir.iterdir()):
        if class_dir.is_dir():
            counts[class_dir.name] = len([item for item in class_dir.iterdir() if item.is_dir() or item.is_file()])
    return counts

def main():

    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)

    if not EXTRACTED_FRAMES_DIR.exists():
        raise FileNotFoundError(
            f"No extracted frames found in {EXTRACTED_FRAMES_DIR}. Please run extract_frames.py first."
        )

    for folder in [SEQUENCES_DIR, TRAIN_DIR, VAL_DIR, TEST_DIR]:
        reset_directory(folder)

    summary = {}

    for class_name in CLASSES:
        class_video_dirs = sorted((EXTRACTED_FRAMES_DIR / class_name).glob("*"))
        random.shuffle(class_video_dirs)

        created_sequences = []

        for video_dir in class_video_dirs:
            frame_paths = sorted(video_dir.glob("*.jpg"))
            indices = sample_frame_indices(len(frame_paths), sequence_length=12)
            selected_frames = [frame_paths[index] for index in indices]

            sequence_dir = SEQUENCES_DIR / class_name / video_dir.name
            copy_sequence_frames(selected_frames, sequence_dir)
            created_sequences.append(sequence_dir)

        train_sequences, val_sequences, test_sequences = split_list(created_sequences)

        for sequence_dir in train_sequences:
            shutil.copytree(sequence_dir, TRAIN_DIR / class_name / sequence_dir.name)
        for sequence_dir in val_sequences:
            shutil.copytree(sequence_dir, VAL_DIR / class_name / sequence_dir.name)
        for sequence_dir in test_sequences:
            shutil.copytree(sequence_dir, TEST_DIR / class_name / sequence_dir.name)

        summary[class_name] = {
            "total_sequences": len(created_sequences),
            "train": len(train_sequences),
            "val": len(val_sequences),
            "test": len(test_sequences),
        }


    print("Sequence preparation is complete.\n")
    print("Train sequence counts:", count_items_per_class(TRAIN_DIR))
    print("Val sequence counts:", count_items_per_class(VAL_DIR))
    print("Test sequence counts:", count_items_per_class(TEST_DIR))


main()

Sequence preparation is complete.

Train sequence counts: {'jump': 105, 'sit': 99}
Val sequence counts: {'jump': 22, 'sit': 21}
Test sequence counts: {'jump': 24, 'sit': 22}


In [32]:
import torch
from pathlib import Path

from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

IMAGE_SIZE = 128
MEAN = (0.5, 0.5, 0.5)
STD = (0.5, 0.5, 0.5)

def get_train_frame_transform():
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ])


def get_eval_frame_transform():
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ])


class VideoSequenceDataset(Dataset):
    """
    One sample is one folder containing ordered frames.
    Sample tensor shape: [T, C, H, W]
    """

    def __init__(self, root_dir: Path, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        self.class_names = []

        for class_dir in sorted(self.root_dir.iterdir()):
            if not class_dir.is_dir():
                continue
            self.class_names.append(class_dir.name)

        class_to_index = {class_name: index for index, class_name in enumerate(self.class_names)}

        for class_name in self.class_names:
            for sequence_dir in sorted((self.root_dir / class_name).iterdir()):
                if sequence_dir.is_dir():
                    self.samples.append((sequence_dir, class_to_index[class_name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sequence_dir, label = self.samples[index]
        frame_paths = sorted(sequence_dir.glob("*.jpg"))

        frames = []
        for frame_path in frame_paths:
            image = Image.open(frame_path).convert("RGB")
            frame_tensor = self.transform(image) if self.transform else image
            frames.append(frame_tensor)

        sequence_tensor = torch.stack(frames, dim=0)
        # Shape is [T, C, H, W], for example [8, 3, 128, 128].
        return sequence_tensor, label


def create_dataloaders(batch_size: int = 2):
    train_dataset = VideoSequenceDataset(TRAIN_DIR, transform=get_train_frame_transform())
    val_dataset = VideoSequenceDataset(VAL_DIR, transform=get_eval_frame_transform())
    test_dataset = VideoSequenceDataset(TEST_DIR, transform=get_eval_frame_transform())

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    sample_sequences, sample_labels = next(iter(train_loader))
    print(f"Sample sequence shape: {sample_sequences.shape}")
    print(f"Sample label shape: {sample_labels.shape}")
    print("Class names:", train_dataset.class_names)
    print("One sample shape [T, C, H, W] =", sample_sequences[0].shape)
    print("One batch shape [B, T, C, H, W] =", sample_sequences.shape)

    return train_loader, val_loader, test_loader, train_dataset.class_names

In [33]:
import torch
import torch.nn as nn


class FrameEncoder(nn.Module):
    """A tiny CNN that extracts features from one frame."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Input frame shape: [batch_size, 3, 128, 128]
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Shape: [batch_size, 16, 64, 64]
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Shape: [batch_size, 32, 32, 32]
            nn.AdaptiveAvgPool2d((1, 1)),
            # Shape: [batch_size, 32, 1, 1]
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        # Shape becomes [batch_size, 32]
        return x


class SimpleVideoCNN(nn.Module):
    """
    Process each frame with the same CNN, then average features across time.
    Input shape: [B, T, C, H, W]
    """

    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.frame_encoder = FrameEncoder()
        self.classifier = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        batch_size, time_steps, channels, height, width = x.shape

        x = x.view(batch_size * time_steps, channels, height, width)
        frame_features = self.frame_encoder(x)
        frame_features = frame_features.view(batch_size, time_steps, -1)
        # Shape is now [B, T, feature_dim].

        video_features = frame_features.mean(dim=1)
        # Temporal average pooling makes shape [B, feature_dim].

        logits = self.classifier(video_features)
        return logits


def print_model_summary(model: nn.Module):
    total_params = sum(parameter.numel() for parameter in model.parameters())
    trainable_params = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )

    print("\nModel structure:")
    print(model)
    print(f"\nTotal parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

In [41]:
import argparse

import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path

ROOT = Path.cwd() 
OUTPUTS_DIR = ROOT / "outputs"
MODELS_DIR = OUTPUTS_DIR / "models"



def run_one_epoch(model, dataloader, criterion, optimizer, device, training: bool):
    if training:
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for sequences, labels in dataloader:
            sequences = sequences.to(device)
            labels = labels.to(device)

            if training:
                optimizer.zero_grad()

            outputs = model(sequences)
            loss = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * labels.size(0)
            predictions = torch.argmax(outputs, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def get_device():
    """
    Return the best available device:
    1. mps  (Apple Silicon GPU, e.g. M2 Mac)
    2. cuda (NVIDIA GPU)
    3. cpu  (fallback)
    """
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

def main():
    parser = argparse.ArgumentParser(description="Train a simple video classifier.")
    parser.add_argument("--epochs", type=int, default=5, help="Number of training epochs.")
    parser.add_argument("--batch_size", type=int, default=2, help="Batch size for DataLoader.")
    parser.add_argument("--learning_rate", type=float, default=0.001, help="Adam learning rate.")
    args = parser.parse_args([])

    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    device = get_device()
    print("Using device:", device)
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    train_loader, val_loader, _, class_names = create_dataloaders(batch_size=args.batch_size)

    model = SimpleVideoCNN(num_classes=len(class_names)).to(device)
    print_model_summary(model)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=args.learning_rate)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    model_save_path = MODELS_DIR / "best_video_model.pth"

    for epoch in range(args.epochs):
        train_loss, train_acc = run_one_epoch(
            model, train_loader, criterion, optimizer, device, training=True
        )
        val_loss, val_acc = run_one_epoch(
            model, val_loader, criterion, optimizer, device, training=False
        )

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch [{epoch + 1}/{args.epochs}] | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), model_save_path)
            print(f"Saved new best model to {model_save_path}")


    print("\nTraining complete.")
    print("Best validation accuracy:", round(best_val_acc, 4))
    print("Saved model to:", model_save_path)
    print("Saved class names to:", MODELS_DIR / "class_names.json")


main()

Using device: mps
Sample sequence shape: torch.Size([2, 12, 3, 128, 128])
Sample label shape: torch.Size([2])
Class names: ['jump', 'sit']
One sample shape [T, C, H, W] = torch.Size([12, 3, 128, 128])
One batch shape [B, T, C, H, W] = torch.Size([2, 12, 3, 128, 128])

Model structure:
SimpleVideoCNN(
  (frame_encoder): FrameEncoder(
    (features): Sequential(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): ReLU()
      (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (6): AdaptiveAvgPool2d(output_size=(1, 1))
    )
  )
  (classifier): Sequential(
    (0): Linear(in_features=32, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=2, bias=True)
  )
)

Total parameters: 6,210
Trainable parameters: 6